# Metrics and reporting stack: an executable tutorial

The visualization stack separates four concerns:

```
WHAT DATA EXISTS        -> MetricSchema
HOW DATA ACCUMULATES    -> Metric / MetricLogger
WHAT DATA TO SELECT     -> Query
HOW IT IS DISPLAYED     -> Reporter
```

The important consequence is that the environment, the inner optimizer and the
outer optimizer do not need backend-specific plotting logic. They expose typed
metrics. Queries select paths from those metrics. A reporter renders the
selected data to Weights & Biases, CSV, TensorBoard or another backend.

This notebook runs every step of that pipeline on synthetic data, without Ray,
without RLlib training and without any network access. It executes in well
under two minutes. It covers:

1. accumulating metrics with `MetricLogger`, and the difference between
   `peek()` and `reduce()`;
2. resolving `Query` objects into labeled `Series`, including wildcards and
   mean ± std grouping;
3. the inner-optimizer schema (`RaySchema`) and the open issue with
   per-episode wildcard queries;
4. the outer-optimizer schema (`ESSchema`) with runtime specialization of the
   `inner` branch;
5. the CSV reporter and how to reload its output with pandas;
6. the three configuration levels of a bilevel run;
7. how to add a metric by subclassing a schema at the right level;
8. a reporting-lifecycle recap and a validation checklist.

In [ ]:
# Make `core` and `examples` importable whatever the kernel's working directory:
# `core` is an installed package, `examples` is only reachable from the repo root.
import sys
from pathlib import Path

try:
    import examples  # noqa: F401
except ModuleNotFoundError:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    repo_root = next(p for p in candidates if (p / "examples").is_dir() and (p / "core").is_dir())
    sys.path.insert(0, str(repo_root))
    import examples  # noqa: F401

print("repo root:", Path(examples.__file__).resolve().parent.parent)

In [ ]:
from __future__ import annotations

import math
import tempfile
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pydantic import Field

from core.adaptors.ray.schema import (
    LearnerSchema,
    MechanismRolloutSchema,
    PerformanceSchema,
    PolicyLearnerSchema,
    RaySchema,
    RolloutSchema,
    SeedRolloutSchema,
    TrainSchema,
)
from core.envs.schema import AgentEnvStepSchema, EpisodeRolloutSchema
from core.metrics.enums import ReduceProtocol
from core.metrics.logger import MetricLogger
from core.metrics.schemas import MetricSchema
from core.optimizers.es.schema import ESCandidateSchema, ESParameterSchema, ESSchema
from core.reporting.base import Reporter
from core.reporting.csv import CSVConfig
from core.reporting.query import Query, Series
from core.reporting.wandb import WandbReporter
from examples.bilevel_fishery import queries
from examples.bilevel_fishery.metric_schema import (
    FisheryAgentMetricSchema,
    FisheryMetricSchema,
)

np.random.seed(0)

## 1. Query paths are tuples

A query points to a metric leaf through a schema tree. Paths are tuples of
field names, so remember the trailing comma of a one-element tuple:

```python
("generation")   # a string -- WRONG
("generation",)  # a one-element tuple -- CORRECT
```

The `Query` API is:

```python
Query(
    title: str,
    x: tuple[str, ...],
    y: tuple[str, ...] | tuple[tuple[str, ...], ...],
    reduce: Literal["none", "mean"] = "none",
    error: Literal["none", "std"] = "none",
)
```

Semantics:

- one `y` path gives one series;
- several `y` paths with `reduce="none"` give several raw series;
- `reduce="mean"` averages pointwise, and `error="std"` adds a ± 1 standard
  deviation band (it requires a reduction);
- at a *dynamic* node (a `dict[ID, MetricSchema]` field) a path component may
  be the wildcard `"*"`, which expands to every runtime key in sorted order.
  With `reduce="mean"`, one wildcard level averages across all matches; two or
  more levels group by the first binding and average over the rest.

Wildcard support is implemented; the sections below exercise it.

In [ ]:
simple_query = Query(title="Fish biomass", x=("iter",), y=("fish_norm",))
print(simple_query)
print("y paths:", simple_query.y_paths, "| wildcards:", simple_query.has_wildcards)

wildcard_query = Query(title="Reward by agent", x=("iter",), y=("by_agent", "*", "reward"))
print("y paths:", wildcard_query.y_paths, "| wildcards:", wildcard_query.has_wildcards)

try:
    Query(title="bad", x=("iter",), y=("fish_norm",), error="std")
except ValueError as e:
    print("ValueError:", e)

## 2. Metric accumulation: `peek()` versus `reduce()`

`MetricLogger.from_schema(schema)` builds one accumulator per schema leaf. The
reducer of a leaf comes from `Field(json_schema_extra={"reduce": ...})` and
defaults to `MEAN`; the available protocols are `MEAN`, `SUM`, `MIN`, `MAX`,
`LAST` and `SERIES`. A `dict[ID, MetricSchema]` field is a dynamic node whose
children are created on first use.

Two ways of reading the logger:

- `peek()` is non-destructive and returns a schema instance whose leaves are
  the raw histories (lists), whatever the reducer. This is what reporters
  consume, because every history is a plottable series.
- `reduce()` is destructive: it applies the reducers, clears the accumulators
  and returns a schema instance with reduced values. Empty leaves reduce to
  `[]` for `SERIES`, `0` for `SUM` and `None` for the other protocols.

We simulate a 20-step fishery episode with three fishers.

In [ ]:
HORIZON = 20
AGENT_IDS = ("utilizer:0", "utilizer:1", "utilizer:2")

env_logger = MetricLogger.from_schema(FisheryMetricSchema)

fish_norm = 0.8
for step in range(HORIZON):
    env_logger.push(("iter",), step)
    env_logger.push(("fish_norm",), fish_norm)
    harvest = 0.05 * fish_norm * len(AGENT_IDS)
    env_logger.push(("H_realized",), harvest)
    for k, agent_id in enumerate(AGENT_IDS):
        reward = 0.05 * fish_norm * (1 + 0.2 * k) + 0.01 * np.random.randn()
        env_logger.push(("by_agent", agent_id, "reward"), reward)
    fish_norm = fish_norm + 0.3 * fish_norm * (1 - fish_norm) - harvest

# peek(): non-destructive, raw histories
peeked = env_logger.peek()
print(type(peeked).__name__)
print("iter      :", peeked.iter)
print("fish_norm :", [round(v, 3) for v in peeked.fish_norm])
print("agents    :", sorted(peeked.by_agent))
print("reward[0] :", [round(v, 3) for v in peeked.by_agent["utilizer:0"].reward][:5], "...")
print("untouched leaf (reward_min):", peeked.reward_min)

In [ ]:
# peek_value(): one leaf, compiled by its reducer, still non-destructive
print("mean fish_norm so far:", round(env_logger.peek_value(("fish_norm",)), 4))

# reduce(): destructive, applies the reducers
reduced = env_logger.reduce()
print("iter (LAST)         :", reduced.iter)
print("fish_norm (MEAN)    :", round(reduced.fish_norm, 4))
print("H_realized (MEAN)   :", round(reduced.H_realized, 4))
print("reward_total (SUM, never pushed) :", reduced.reward_total)
print("reward_min (MIN, never pushed)   :", reduced.reward_min)
print("agent rewards (MEAN):", {a: round(s.reward, 4) for a, s in reduced.by_agent.items()})

# The accumulators are now empty.
after = env_logger.peek()
print("after reduce -> fish_norm history:", after.fish_norm)

The distinction matters most for the outer optimizer: ES plots are cumulative
over generations, so the ES optimizer calls `push_data()` then `peek()` and
hands the complete trajectory to its reporter at every generation. The
environment, on the other hand, reports the horizon series with `peek()` at the
end of an episode and then `reduce()`s the episode into one summary that RLlib
receives (see `core/callbacks.py`).

## 3. From queries to series: a recording reporter

`Reporter.report(metrics)` resolves every registered `Query` against a peeked
schema into a list of `Series(label, x, y, error)` and hands each list to the
backend-specific `_report(query, series)`. Backends never see the metric tree,
only labeled series. A minimal backend that records what it receives is
enough to inspect the resolution.

We rebuild the episode logger (the previous one was reduced) and resolve the
ready-made environment bundles `FISHERY_ENV_QUERIES` and
`FISHERY_ALL_AGENTS_QUERIES` from `examples/bilevel_fishery/queries.py`.

In [ ]:
class RecordingReporter(Reporter):
    # Stores every (query, series) pair it receives and prints a one-line summary.

    def __init__(self) -> None:
        self.reports: list[tuple[Query, list[Series]]] = []

    def _report(self, query: Query, series: list[Series]) -> None:
        self.reports.append((query, series))
        for s in series:
            band = " ± std" if s.error is not None else ""
            print(f"  [{query.title}] {s.label}: {len(s.y)} points{band}")

    def close(self) -> None:
        pass

    def find(self, title: str) -> list[Series]:
        return next(series for q, series in self.reports if q.title == title)


def simulate_episode(horizon: int, agent_ids: tuple[str, ...]) -> FisheryMetricSchema:
    logger = MetricLogger.from_schema(FisheryMetricSchema)
    fish_norm = 0.8
    for step in range(horizon):
        harvest = 0.05 * fish_norm * len(agent_ids)
        logger.push(("iter",), step)
        logger.push(("fish_norm",), fish_norm)
        logger.push(("fish_norm_next",), fish_norm - harvest)
        logger.push(("fish_stock",), 5_000.0 * fish_norm)
        logger.push(("fish_stock_next",), 5_000.0 * (fish_norm - harvest))
        logger.push(("growth",), 0.3 * fish_norm * (1 - fish_norm))
        logger.push(("growth_noise",), 0.01 * np.random.randn())
        logger.push(("H_attempted",), harvest * 1.1)
        logger.push(("H_realized",), harvest)
        logger.push(("allowed_harvest",), 0.15)
        logger.push(("total_usage_norm",), harvest / 0.15)
        logger.push(("quota_stress",), max(0.0, harvest / 0.15 - 1))
        rewards = []
        for k, agent_id in enumerate(agent_ids):
            reward = 0.05 * fish_norm * (1 + 0.2 * k) + 0.01 * np.random.randn()
            rewards.append(reward)
            logger.push(("by_agent", agent_id, "reward"), reward)
            logger.push(("by_agent", agent_id, "delivered_harvest"), harvest / len(agent_ids))
        logger.push(("reward_mean",), float(np.mean(rewards)))
        fish_norm = fish_norm + 0.3 * fish_norm * (1 - fish_norm) - harvest
    return logger.peek()


episode = simulate_episode(HORIZON, AGENT_IDS)

env_reporter = RecordingReporter()
env_reporter.schema = FisheryMetricSchema
env_reporter.add_query(*queries.FISHERY_ENV_QUERIES, *queries.FISHERY_ALL_AGENTS_QUERIES)
env_reporter.report(episode)

Three things to read in that output. The plain queries produce one series
labeled by the metric path. `Reward — all agents` expands the wildcard into one
series per agent, labeled `by_agent/<id>/reward` in sorted id order. `Mean
reward across agents ±1 std` has a single wildcard level and `reduce="mean"`,
so the three agents are averaged into one series (labeled by the query title)
carrying a standard deviation per point.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

for s in env_reporter.find("Fish biomass"):
    axes[0].plot(s.x, s.y, marker="o", ms=3, label=s.label)
axes[0].set(title="Fish biomass", xlabel="iter (env step)", ylabel="fish_norm")

for s in env_reporter.find("Reward — all agents"):
    axes[1].plot(s.x, s.y, marker="o", ms=3, label=s.label)
axes[1].set(title="Reward — all agents (wildcard)", xlabel="iter (env step)")
axes[1].legend(fontsize=8)

(mean_series,) = env_reporter.find("Mean reward across agents ±1 std")
y, err = np.asarray(mean_series.y), np.asarray(mean_series.error)
axes[2].fill_between(mean_series.x, y - err, y + err, alpha=0.25, label="± 1 std")
axes[2].plot(mean_series.x, y, marker="o", ms=3, label=mean_series.label)
axes[2].set(title="Mean reward across agents", xlabel="iter (env step)")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

The W&B backend renders the same `Series` as a Plotly figure: one
`lines+markers` trace per series, preceded by a filled band when the series
carries a standard deviation. `WandbReporter._figure(query, series)` builds
that figure without creating a run, so it can be inspected offline.

In [ ]:
(mean_query,) = [q for q in queries.FISHERY_ALL_AGENTS_QUERIES if q.reduce == "mean"]
fig = WandbReporter._figure(mean_query, env_reporter.find(mean_query.title))
fig.update_layout(title=mean_query.title, xaxis_title="/".join(mean_query.x), height=350)
print("traces:", [trace.name for trace in fig.data])
fig.show()

## 4. Where reporting is configured

There are three distinct reporting levels in a bilevel run, and one reporter
backend shared by all of them.

**A. Environment level.** The environment schema and its horizon queries are
declared on the inner optimizer config:

```python
APPOptimizerConfig().environment(
    env=FisheryRegulatedEnv,
    ...,
    schema=FisheryMetricSchema,
    queries=FISHERY_ENV_QUERIES + FISHERY_ALL_AGENTS_QUERIES,
)
```

**B. Inner optimizer level.** The RLlib result schema and its per-iteration
queries:

```python
APPOptimizerConfig().reporting(schema=RaySchema, queries=RAY_QUERIES)
```

**C. Outer optimizer level.** The ES schema and its per-generation queries:

```python
ESConfig().reporting(schema=ESSchema, queries=ES_QUERIES + ...)
```

**Backend.** One `ReporterConfig` is attached at the bilevel level and copied
to every owner; `build(label=...)` instantiates the backend for one owner:

```python
BilevelConfig().reporter(config=WandbConfig(project="bilevel"))   # or CSVConfig(...)
```

Reporting is optional: without a reporter config nothing is rendered, and
without a schema nothing is logged. Section 9 inspects the complete
configuration of `examples/bilevel_fishery/debug.py`.

## 5. Schema hierarchy

The schema level determines the meaning and the path of a metric.

```
MetricSchema  (carries `iter`, reducer LAST)
│
├── Environment
│   ├── EpisodeRolloutSchema
│   │   └── by_agent: dict[AgentID, AgentEnvStepSchema]
│   └── FisheryMetricSchema(EpisodeRolloutSchema)
│       └── by_agent: dict[AgentID, FisheryAgentMetricSchema(AgentEnvStepSchema)]
│
├── RaySchema
│   ├── train: TrainSchema
│   │   ├── rollout: RolloutSchema
│   │   │   ├── aggregate: EpisodeRolloutSchema
│   │   │   └── by_mechanism: dict -> MechanismRolloutSchema
│   │   │       └── by_seed: dict -> SeedRolloutSchema
│   │   │           └── by_episode: dict -> EpisodeRolloutSchema
│   │   ├── learner: LearnerSchema
│   │   │   └── by_policy: dict -> PolicyLearnerSchema
│   │   └── performance: PerformanceSchema
│   └── eval: EvalSchema (rollout, performance)
│
└── ESSchema
    ├── generation, sigma, population_size, fitness_mean, fitness_best,
    │   best_mechanism_idx, best_fitness_global   (all SERIES)
    ├── by_mechanism: dict -> ESCandidateSchema
    │   ├── fitness (SERIES)
    │   └── by_parameter: dict -> ESParameterSchema.value (SERIES)
    ├── search_mean / global_best / generation_best: dict[ParameterName, ESParameterSchema]
    └── inner: Optional[MetricSchema]   (runtime subtype, e.g. RaySchema)
```

The logger supports runtime schema specialization: pushing a *subclass* of the
declared schema at a nested node rebuilds that sub-tree with the subclass.
This is how `EpisodeRolloutSchema` becomes `FisheryMetricSchema` under
`by_episode`, and how `ESSchema.inner`, declared as the generic
`MetricSchema`, ends up holding a `RaySchema`. The runtime subtype must be a
subclass of the declared one; the root schema itself is not specialized
(`push_data` at the root requires the exact declared type).

In [ ]:
seed_logger = MetricLogger.from_schema(SeedRolloutSchema)
for step in range(3):
    seed_logger.push_data(
        SeedRolloutSchema(
            by_episode={
                "e0": FisheryMetricSchema(
                    fish_norm=0.8 - 0.1 * step,
                    by_agent={"utilizer:0": FisheryAgentMetricSchema(requested_harvest=0.02)},
                )
            }
        )
    )

peeked = seed_logger.peek()
e0 = peeked.by_episode["e0"]
print("declared by_episode value type :", SeedRolloutSchema.model_fields["by_episode"].annotation)
print("runtime by_episode value type  :", type(e0).__name__)
print("fish_norm history              :", e0.fish_norm)
print("runtime by_agent value type    :", type(e0.by_agent["utilizer:0"]).__name__)
print("requested_harvest history      :", e0.by_agent["utilizer:0"].requested_harvest)

## 6. Inner optimizer: `RaySchema`

The Ray adaptor converts each RLlib `ResultDict` into a `RaySchema` (see
`core/adaptors/ray/utils.py` and `tests/adaptors/test_ray_result_builders.py`),
pushes it into the inner logger together with the training iteration `iter`,
and reports. We build two synthetic payloads by hand. The `by_mechanism ->
by_seed -> by_episode` grouping receives `EpisodeRolloutSchema` instances; in a
real run those are the reduced `FisheryMetricSchema` episodes.

In [ ]:
def episode_schema(mechanism_id: int, seed: int, reward: float) -> EpisodeRolloutSchema:
    return EpisodeRolloutSchema(mechanism_id=mechanism_id, seed=seed, reward_mean=reward)


def ray_payload(it: int, episode_ids: dict[str, tuple[int, int, float]]) -> RaySchema:
    # episode_ids maps episode id -> (mechanism, seed, reward_mean)
    by_mechanism: dict[str, MechanismRolloutSchema] = {}
    for episode_id, (mechanism, seed, reward) in episode_ids.items():
        mech = by_mechanism.setdefault(str(mechanism), MechanismRolloutSchema())
        seed_node = mech.by_seed.setdefault(str(seed), SeedRolloutSchema())
        seed_node.by_episode[episode_id] = episode_schema(mechanism, seed, reward)
    rewards = [r for _, _, r in episode_ids.values()]
    return RaySchema(
        train=TrainSchema(
            rollout=RolloutSchema(
                aggregate=EpisodeRolloutSchema(
                    reward_mean=float(np.mean(rewards)),
                    reward_min=min(rewards),
                    reward_max=max(rewards),
                    episode_len_mean=20.0,
                    episode_len_min=20.0,
                    episode_len_max=20.0,
                    num_episodes=len(rewards),
                ),
                by_mechanism=by_mechanism,
            ),
            learner=LearnerSchema(
                by_policy={
                    "fisher_policy": PolicyLearnerSchema(
                        total_loss=1.0 / it, policy_entropy=1.5 - 0.1 * it
                    ),
                    "regulator_policy": PolicyLearnerSchema(
                        total_loss=0.5 / it, policy_entropy=1.0 - 0.1 * it
                    ),
                }
            ),
            performance=PerformanceSchema(
                env_steps_this_iter=80.0,
                env_steps_lifetime=80.0 * it,
                training_iteration_s=1.0,
                sample_s=0.3,
                learner_update_s=0.2,
            ),
        )
    )


ray_logger = MetricLogger.from_schema(RaySchema)
# Episode ids are unique per episode in RLlib (e1..e4 in iteration 1, e5..e8 in
# iteration 2), while mechanism and seed ids repeat across iterations.
ray_logger.push(("iter",), 1)
ray_logger.push_data(ray_payload(1, {"e1": (0, 100, 1.0), "e2": (0, 200, 2.0), "e3": (1, 100, 3.0), "e4": (1, 200, 4.0)}))
ray_logger.push(("iter",), 2)
ray_logger.push_data(ray_payload(2, {"e5": (0, 100, 1.5), "e6": (0, 200, 2.5), "e7": (1, 100, 3.5), "e8": (1, 200, 4.5)}))

ray_metrics = ray_logger.peek()
print("iter                         :", ray_metrics.iter)
print("train.rollout.aggregate.reward_mean :", ray_metrics.train.rollout.aggregate.reward_mean)
print("mechanisms                   :", sorted(ray_metrics.train.rollout.by_mechanism))
print("episodes under m0/s100       :", sorted(ray_metrics.train.rollout.by_mechanism["0"].by_seed["100"].by_episode))

In [ ]:
policy_wildcard = Query(
    title="Total loss — all policies",
    x=("iter",),
    y=("train", "learner", "by_policy", "*", "total_loss"),
)

ray_reporter = RecordingReporter()
ray_reporter.schema = RaySchema
ray_reporter.add_query(*queries.RAY_ROLLOUT_QUERIES, *queries.RAY_PERFORMANCE_QUERIES, policy_wildcard)
ray_reporter.report(ray_metrics)

for s in ray_reporter.find("Train reward"):
    print(s.label, "->", s.y)
for s in ray_reporter.find(policy_wildcard.title):
    print(s.label, "->", s.y)

### Open issue: `by_episode/*` does not align with the `iter` axis

Episode ids are unique per episode, whereas the inner logger accumulates one
value per training iteration. After two iterations the `iter` series has
length 2, but every `by_episode/<id>/reward_mean` history has length 1, so a
wildcard query over episodes fails the length check of the base reporter.
This is the point left open in `TODO.md` (sections 3 and 4): aligning episodes
with iterations needs an episode-to-iteration key, and that decision has not
been taken. `SeedRolloutSchema.aggregate` will **not** be added; aggregation
across seeds and episodes happens at query resolution.

In [ ]:
episode_wildcard = Query(
    title="Reward by mechanism ±1 std across seeds and episodes",
    x=("iter",),
    y=("train", "rollout", "by_mechanism", "*", "by_seed", "*", "by_episode", "*", "reward_mean"),
    reduce="mean",
    error="std",
)
try:
    ray_reporter._resolve_query(ray_metrics, episode_wildcard)
except ValueError as e:
    print("ValueError:", e)

To see what the grouping semantics would give once episodes are keyed by their
slot within an iteration, we log the same two iterations with stable episode
ids (`e0`, one episode per mechanism × seed). The two wildcard levels that
remain after the concrete `by_episode/e0` group by the first binding (the
mechanism) and average over the second (the seed). This is an illustration of
the query semantics, not of an implemented adaptor behavior.

In [ ]:
aligned_logger = MetricLogger.from_schema(RaySchema)
aligned_logger.push(("iter",), 1)
aligned_logger.push_data(ray_payload(1, {"e0": (0, 100, 1.0)}))
aligned_logger.push_data(ray_payload(1, {"e0": (0, 200, 2.0)}))
aligned_logger.push_data(ray_payload(1, {"e0": (1, 100, 3.0)}))
aligned_logger.push_data(ray_payload(1, {"e0": (1, 200, 4.0)}))
aligned_logger.push(("iter",), 2)
aligned_logger.push_data(ray_payload(2, {"e0": (0, 100, 1.5)}))
aligned_logger.push_data(ray_payload(2, {"e0": (0, 200, 2.5)}))
aligned_logger.push_data(ray_payload(2, {"e0": (1, 100, 3.5)}))
aligned_logger.push_data(ray_payload(2, {"e0": (1, 200, 4.5)}))

grouped = Query(
    title="Reward by mechanism ±1 std across seeds",
    x=("iter",),
    y=("train", "rollout", "by_mechanism", "*", "by_seed", "*", "by_episode", "e0", "reward_mean"),
    reduce="mean",
    error="std",
)
for s in ray_reporter._resolve_query(aligned_logger.peek(), grouped):
    print(f"group {s.label}: mean={s.y} std={s.error}")

## 7. Outer optimizer: `ESSchema`

One completed generation becomes one `ESSchema` payload. Its scalar fields
are `SERIES`, so the ES logger, which stays alive across generations, grows
each history by one value per `push_data()`. `by_mechanism` holds the fitness
and parameter values of every candidate, `search_mean`, `global_best` and
`generation_best` are keyed by parameter name, and `inner` receives the
reduced metrics of the inner optimizer, here a `RaySchema`.

We push three generations for four candidates and two parameters.

In [ ]:
PARAMETERS = ("fixed_quota", "restoration_subsidy")
NUM_CANDIDATES = 4


def es_payload(generation: int, mean: np.ndarray, sigma: float, best_so_far: float) -> ESSchema:
    population = mean + sigma * np.random.randn(NUM_CANDIDATES, len(PARAMETERS))
    # A smooth synthetic fitness landscape peaking at (0.5, 0.1).
    fitness = -(((population - np.array([0.5, 0.1])) ** 2).sum(axis=1)) + 1.0
    best = int(np.argmax(fitness))
    best_global = max(best_so_far, float(fitness[best]))
    inner = RaySchema(
        train=TrainSchema(
            rollout=RolloutSchema(aggregate=EpisodeRolloutSchema(reward_mean=float(fitness.mean()))),
            learner=LearnerSchema(),
            performance=PerformanceSchema(),
        )
    )
    return ESSchema(
        generation=generation,
        sigma=sigma,
        population_size=NUM_CANDIDATES,
        fitness_mean=float(fitness.mean()),
        fitness_best=float(fitness[best]),
        best_mechanism_idx=best,
        best_fitness_global=best_global,
        by_mechanism={
            str(i): ESCandidateSchema(
                fitness=float(fitness[i]),
                by_parameter={
                    name: ESParameterSchema(value=float(population[i, j]))
                    for j, name in enumerate(PARAMETERS)
                },
            )
            for i in range(NUM_CANDIDATES)
        },
        search_mean={name: ESParameterSchema(value=float(mean[j])) for j, name in enumerate(PARAMETERS)},
        global_best={name: ESParameterSchema(value=float(population[best, j])) for j, name in enumerate(PARAMETERS)},
        generation_best={name: ESParameterSchema(value=float(population[best, j])) for j, name in enumerate(PARAMETERS)},
        inner=inner,
    )


es_logger = MetricLogger.from_schema(ESSchema)
mean, sigma, best_global = np.array([0.7, 0.3]), 0.15, -math.inf
for generation in range(1, 4):
    payload = es_payload(generation, mean, sigma, best_global)
    es_logger.push_data(payload)
    best_global = payload.best_fitness_global
    mean = mean + 0.5 * (np.array([0.5, 0.1]) - mean)  # move toward the optimum

es_metrics = es_logger.peek()
print("generation     :", es_metrics.generation)
print("fitness_mean   :", [round(v, 3) for v in es_metrics.fitness_mean])
print("fitness_best   :", [round(v, 3) for v in es_metrics.fitness_best])
print("candidate 0 fitness :", [round(v, 3) for v in es_metrics.by_mechanism["0"].fitness])
print("inner is a", type(es_metrics.inner).__name__, "->", es_metrics.inner.train.rollout.aggregate.reward_mean)

In [ ]:
es_reporter = RecordingReporter()
es_reporter.schema = ESSchema
es_reporter.add_query(
    *queries.ES_QUERIES,
    *queries.es_parameter_queries(PARAMETERS),
    *queries.es_candidate_fitness_queries(NUM_CANDIDATES),
    *queries.es_parameter_fitness_queries(PARAMETERS),
    Query(
        title="Inner reward per generation",
        x=("generation",),
        y=("inner", "train", "rollout", "aggregate", "reward_mean"),
    ),
)
es_reporter.report(es_metrics)

`Candidate fitness (all candidates)` and `Mean candidate fitness ±1 std` use
the same wildcard path `by_mechanism/*/fitness`; the first keeps one series per
candidate, the second averages them into one series with a standard deviation.
`Fitness vs <parameter>` binds the wildcard on both sides: candidate `k`'s
parameter values are paired with candidate `k`'s fitness, never with another
candidate's, so each series is one candidate's trajectory in the (parameter,
fitness) plane.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

for s in es_reporter.find("Candidate fitness (all candidates)"):
    axes[0].plot(s.x, s.y, "o", ms=5, alpha=0.7, label=s.label)
(mean_fit,) = es_reporter.find("Mean candidate fitness ±1 std")
y, err = np.asarray(mean_fit.y), np.asarray(mean_fit.error)
axes[0].fill_between(mean_fit.x, y - err, y + err, alpha=0.2)
axes[0].plot(mean_fit.x, y, "-", lw=2, label="mean ± 1 std")
axes[0].set(title="Candidate fitness vs mean", xlabel="generation", ylabel="fitness")
axes[0].legend(fontsize=7)

for s in es_reporter.find("Fitness over generations"):
    axes[1].plot(s.x, s.y, marker="o", label=s.label)
axes[1].set(title="Fitness over generations", xlabel="generation")
axes[1].legend(fontsize=8)

for s in es_reporter.find("Fitness vs fixed_quota"):
    axes[2].plot(s.x, s.y, "o-", ms=4, alpha=0.8, label=s.label)
axes[2].axvline(0.5, ls="--", c="grey", lw=1)
axes[2].set(title="Fitness vs fixed_quota (x/y bound per candidate)", xlabel="fixed_quota")
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.show()

for s in es_reporter.find("Fitness vs fixed_quota"):
    print(s.label, "x =", [round(v, 3) for v in s.x], "y =", [round(v, 3) for v in s.y])

## 8. CSV reporter

`CSVConfig.build(label=...)` creates a `CSVReporter` writing one long-form file
per query under `output_dir/<project>/<world>-<label>/`. Each row is
`(query, x, series, value, error)`; the file is rewritten on every report, so
it always holds the latest complete trajectory. The world name and the number
of outer iterations are normally injected by `BilevelConfig.build_optimizer()`;
here we set them by hand.

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="viz_tutorial_"))

csv_cfg = CSVConfig(project="tutorial", output_dir=str(tmp_dir))
csv_cfg.world = "fishery_world"
csv_cfg.outer_iters = 3

csv_reporter = csv_cfg.build(label="env")
csv_reporter.schema = FisheryMetricSchema
csv_reporter.add_query(*queries.FISHERY_ENV_QUERIES, *queries.FISHERY_ALL_AGENTS_QUERIES)
csv_reporter.report(episode)
csv_reporter.close()

print("output dir:", csv_reporter.output_dir.relative_to(tmp_dir))
for path in sorted(csv_reporter.output_dir.iterdir()):
    print("  ", path.name)

In [ ]:
mean_reward_csv = pd.read_csv(csv_reporter.path_for(mean_query))
display(mean_reward_csv.head())

all_agents_csv = pd.read_csv(csv_reporter.path_for(queries.FISHERY_ALL_AGENTS_QUERIES[0]))
wide = all_agents_csv.pivot(index="x", columns="series", values="value")
display(wide.head())
wide.plot(figsize=(7, 3.2), marker="o", ms=3, title="Reloaded from CSV: reward per agent")
plt.xlabel("iter (env step)")
plt.tight_layout()
plt.show()

## 9. The three configuration levels on the fishery example

`examples/bilevel_fishery/debug.py::build_config` assembles the complete
bilevel configuration. Building the config does not start Ray and does not
touch the network; only `build_optimizer()` would. We build it with the CSV
reporter and inspect what each level registered.

In [ ]:
from examples.bilevel_fishery.debug import build_config, parse_args

cfg = build_config(parse_args(["--reporter", "csv", "--outer-iters", "3", "--num-agents", "3"]))

print("backend           :", type(cfg.reporter_cfg).__name__, "->", cfg.reporter_cfg.output_dir)
print("world             :", cfg.world_name)

outer, inner = cfg.outer_cfg, cfg.inner_cfg
print()
print("A. environment    :", inner._reporting_schema_env.__name__, f"({len(inner._reporting_queries_env)} queries)")
for q in inner._reporting_queries_env[:3]:
    print("     ", q.title, q.y)
print("      ...")
print("B. inner optimizer:", inner._reporting_schema.__name__, f"({len(inner._reporting_queries)} queries)")
for q in inner._reporting_queries:
    print("     ", q.title)
print("C. outer optimizer:", outer._reporting_schema.__name__, f"({len(outer._reporting_queries)} queries)")
for q in outer._reporting_queries:
    print("     ", q.title, "| x =", "/".join(q.x), "| reduce =", q.reduce)

## 10. Adding a metric: schema extension rules

The schema level decides the path of a metric and who produces it.

- **Shared environment metric** (describes the episode): subclass
  `EpisodeRolloutSchema`, as `FisheryMetricSchema` does. Path: `("fish_norm",)`
  in the environment logger, or
  `("train", "rollout", "by_mechanism", m, "by_seed", s, "by_episode", e, "fish_norm")`
  once nested inside `RaySchema`.
- **Per-agent metric**: subclass `AgentEnvStepSchema` and override `by_agent`
  in the episode schema with `dict[AgentID, YourAgentSchema]`. Path:
  `("by_agent", "*", "requested_harvest")`.
- **Learner metric**: add a field to `PolicyLearnerSchema`; the policy
  dimension stays a `dict[PolicyID, PolicyLearnerSchema]`, never one static
  field per policy. Path: `("train", "learner", "by_policy", "*", "gradient_norm")`.
- **Performance metric**: add a field to `PerformanceSchema`. Path:
  `("train", "performance", "env_steps_throughput")`.
- **ES generation metric**: add a `SERIES` field to `ESSchema` when the plot
  needs the full outer history. Candidate metrics go to `ESCandidateSchema`,
  parameter metrics to `ESParameterSchema`.
- **Inner optimizer under ES**: keep `inner: Optional[MetricSchema] = None`
  and rely on runtime subtype binding; do not hard-code `RaySchema`.

Do not create a query for a field that is not represented in a schema. The
example below adds an environment-level `price` and a per-agent `effort` to
the fishery schemas and queries them with a wildcard.

In [ ]:
class TutorialAgentSchema(FisheryAgentMetricSchema):
    effort: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.MEAN}
    )


class TutorialFisherySchema(FisheryMetricSchema):
    price: Optional[float] = Field(
        default=None, json_schema_extra={"reduce": ReduceProtocol.MEAN}
    )
    by_agent: dict[str, TutorialAgentSchema] = Field(default_factory=dict)


tutorial_logger = MetricLogger.from_schema(TutorialFisherySchema)
for step in range(5):
    tutorial_logger.push(("iter",), step)
    tutorial_logger.push(("price",), 10.0 + step)
    for k, agent_id in enumerate(AGENT_IDS):
        tutorial_logger.push(("by_agent", agent_id, "effort"), 0.1 * (k + 1) + 0.01 * step)

tutorial_reporter = RecordingReporter()
tutorial_reporter.add_query(
    Query(title="Price", x=("iter",), y=("price",)),
    Query(title="Effort — all agents", x=("iter",), y=("by_agent", "*", "effort")),
    Query(title="Mean effort ±1 std", x=("iter",), y=("by_agent", "*", "effort"), reduce="mean", error="std"),
)
tutorial_reporter.report(tutorial_logger.peek())

(s,) = tutorial_reporter.find("Mean effort ±1 std")
print("mean effort:", [round(v, 3) for v in s.y], "std:", [round(v, 3) for v in s.error])

# Every declared subclass is still accepted where the base is declared: a
# SeedRolloutSchema logger specializes by_episode with the new schema at runtime.
seed_logger = MetricLogger.from_schema(SeedRolloutSchema)
seed_logger.push_data(SeedRolloutSchema(by_episode={"e0": TutorialFisherySchema(price=12.0)}))
print("by_episode runtime type:", type(seed_logger.peek().by_episode["e0"]).__name__)

## 11. Reporting lifecycle

**Environment.** At episode completion (`core/callbacks.py`):

```python
metrics = env.logger.peek()
env.reporter.report(metrics)     # horizon plots, before destructive reduction
reduced = env.logger.reduce()    # one episode summary, handed to RLlib
```

**Ray/RLlib.** The adaptor turns each `ResultDict` into a `RaySchema`,
pushes it into its logger with the training iteration as `iter`, and reports
the accumulated view; `reduce()` at the end of the inner run yields the
`RaySchema` that becomes `ESSchema.inner`.

**ES.** One completed generation becomes one `ESSchema` payload
(`core/optimizers/es/optimizer.py`):

```python
self.logger.push_data(metrics)
metrics = self.logger.peek()
self.reporting.report(metrics)
```

The reporter receives the complete trajectory at each generation and does not
need to own a second history cache: the metric logger owns history.

**Backends.** `WandbReporter` logs one Plotly figure per query
(`plots/<sanitized title>`) to a run created lazily on the first report;
`CSVReporter` writes one long-form file per query; `TensorBoardReporter`
(optional `tensorboard` extra) writes one scalar tag per series, indexed by an
integer x. All three consume the same `Series` contract and none of them
changes schema or query semantics.

## 12. Validation checklist

For a deterministic small integration run (4 candidates, 2 or 3 seeds, a short
horizon, short inner training, at least 3 generations), check:

**Environment** — fish biomass, fish stock and next stock, growth and growth
noise, attempted / realized / allowed harvest, quota stress, total normalized
usage, mean reward; per-agent reward and delivered harvest; mean reward across
agents ± 1 std.

**Inner optimizer** — rollout reward mean/min/max, episode length, episode
count; environment steps and timing; total loss and entropy per policy
(wildcard over `by_policy`). Per-mechanism series averaged across seeds remain
blocked by the episode-alignment issue of section 6.

**ES** — fitness mean, generation best and global best over generations;
candidate fitness markers and their mean ± 1 std; sigma and population size;
search mean, global-best and generation-best value per optimized parameter;
fitness versus parameter with x/y bound per candidate. Still unsupported by
the `Query(x, y)` abstraction: scatter points colored by generation and the
cumulative parallel-coordinates plot, which need a dedicated multidimensional
query.

**Design rule for future metrics.** Before adding a visualization, decide
(1) which entity owns the metric (episode, agent, policy, optimizer
performance, ES generation / candidate / parameter); (2) which reducer applies
(`SERIES` when the history itself is the metric, otherwise `MEAN`, `MIN`,
`MAX`, `SUM` or `LAST`); (3) whether the key exists at runtime (static field
versus `dict[ID, MetricSchema]` queried with `"*"`); (4) whether the plot is a
line/scatter expressible as a `Query` or needs multidimensional support. This
keeps collection, aggregation, selection and presentation separate.